# State

In [1]:
# Import some technical indicators from the library
from pyti.exponential_moving_average import exponential_moving_average as ema
from pyti.aroon import aroon_down as aroondwn
from pyti.aroon import aroon_up as aroonup
from pyti.bollinger_bands import lower_bollinger_band as bblow
from pyti.bollinger_bands import middle_bollinger_band as bbmid
from pyti.bollinger_bands import upper_bollinger_band as bbup
from pyti.momentum import momentum as mom
from pyti.simple_moving_average import simple_moving_average as sma
from pyti.stochastic import percent_d as stochd
from pyti.stochastic import percent_k as stochk
from pyti.stochrsi import stochrsi as stochrsi
from pyti.stochrsi import relative_strength_index as rsi
from pyti.williams_percent_r import williams_percent_r as willr

In [93]:
import numpy as np
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
from datetime import datetime, timedelta
from keras import Sequential
from keras import Input
from keras.layers import Dense
import keras.backend as K
import traceback
import pickle
from copy import deepcopy
import warnings
import os
import random
import tensorflow as tf
warnings.filterwarnings("ignore", category=RuntimeWarning)

%matplotlib inline

def reward_fun_dummy():
    pass

reward_function = reward_fun_dummy

# State

- 5 minute (45)
    - Open, high, low, and close for last 10 days (40)
    - SMA Crossover
    - RSI
    - Mom
    - Aroon Up
    - Aroon Down
- 1 hour (Same as above) (45)
- 1 day (Same as above) (45)
- Position (1)
- Current Date (1)
- Current Time (1)

In [86]:
class Game(object):

    def __init__(self, df,bars1d,bars1h,reward_function, lkbk=20, init_idx=None):
        self.df = df
        self.lkbk = lkbk
        self.trade_len = 0
        self.stop_pnl = None
        self.bars1d =  bars1d
        self.bars1h =  bars1h
        self.is_over = False
        self.reward = 0
        self.pnl_sum = 0
        self.init_idx = init_idx
        self.reward_function = reward_function
        self.reset()
        self.curr_time = self.df.index[self.init_idx]
        tm_lst = list(map(float,str(self.curr_time.time()).split(':')[:2]))
        self._time_of_day = (tm_lst[0]*60 + tm_lst[1])/(24*60)
        self._day_of_week  = self.curr_time.weekday()/6
        
        self.position = 0

    def _assemble_state(self):
        '''Here we can add secondary features such as indicators and times to our current state.
        First, we create candlesticks for different bar sizes of 5mins, 1hr and 1d.
        We then add some state variables such as time of day, day of week and position.
        Next several indicators are added and subsequently z-scored.
        '''
    
        """---Adding State Variables---"""
        self.state = np.array([])
    
        """---Adding Candlesticks---"""
        self._get_last_N_timebars()
        bars = [self.last5m,self.last1h,self.last1d]
        state_bars = []
        candles = {j:{k:np.array([]) for k in ['open','high','low','close']} for j in range(len(bars))}
        for j,bar in enumerate(bars):
            for col in ['open','high','low','close']:
                candles[j][col] = np.asarray(bar[col])
                state_bars += (list(np.asarray(bar[col]))[-10:])
    
        self.state = np.append(self.state,state_bars)
    
        """---Adding Techincal Indicators---"""
        for c in candles:
            try:
                sma1 = sma(candles[c]['close'],self.lkbk-1)[-1]
                sma2 = sma(candles[c]['close'],self.lkbk-8)[-1]
                self.state = np.append(self.state,(sma1-sma2)/sma2)
                self.state = np.append(self.state,rsi(candles[c]['close'],self.lkbk-1)[-1])
                self.state = np.append(self.state,mom(candles[c]['close'],self.lkbk-1)[-1])
                self.state = np.append(self.state,aroonup(candles[c]['high'],self.lkbk-3)[-1])
                self.state = np.append(self.state,aroondwn(candles[c]['low'],self.lkbk-3)[-1])
            except: print(traceback.format_exc())
    
        """---Normalizing Candlesticks---"""
        self.state = (np.array(self.state)-np.mean(self.state,axis=0))/np.std(self.state,axis=0)
    
        self.state = np.append(self.state,self.position)
        self.state = np.append(self.state,self._time_of_day)
        self.state = np.append(self.state,self._day_of_week)
        #print(list(map(len,[self.last5m,self.last1h,self.last1d])))
    
    def _get_last_N_timebars(self):
        '''This function gets the timebars for the 5m, 1hr and 1d resolution based
        on the lookback we've specified.
        '''
        wdw5m = 9
        wdw1h = np.ceil(self.lkbk*15/24.)
        wdw1d = np.ceil(self.lkbk*17)
        
    
        """---creating the candlesticks based on windows---"""
        self.last5m = self.df[self.curr_time-timedelta(wdw5m):self.curr_time].iloc[-self.lkbk:]
        self.last1h = self.bars1h[self.curr_time-timedelta(wdw1h):self.curr_time].iloc[-self.lkbk:]
        self.last1d = self.bars1d[self.curr_time-timedelta(wdw1d):self.curr_time].iloc[-self.lkbk:]


    def reset(self):
        pass
        

In [87]:
import pickle
df = pickle.load(open("MLT-07_data_PriceData.pick",'rb')).dropna()

df = df.iloc[-10000:,:]

df.head()

,open,high,low,close,volume
Time,,,,,
2014-05-23 19:35:00,87.248,87.252,87.244,87.248,1716.0
2014-05-23 19:40:00,87.248,87.256,87.246,87.254,1716.0
2014-05-23 19:45:00,87.254,87.260,87.252,87.258,5195.4
2014-05-23 19:50:00,87.258,87.264,87.256,87.262,5419.2
2014-05-23 19:55:00,87.264,87.268,87.262,87.266,6959.6


In [88]:
bars5m = df
bars1h = df['close'].resample('1h',label='right',closed='right').ohlc().dropna()
bars1d = df['close'].resample('1D',label='right',closed='right').ohlc().dropna()

In [96]:
# df: This is the data frame with the market data
# lkbk: This is the lookback period, eg. a value of 10 means 10mins, 10hrs and 10days!

env = Game(df, bars1d,bars1h,reward_function,lkbk=10, init_idx=8000)
env._assemble_state()
env.state

array([ 0.24866375,  0.25121293,  0.25268877,  0.25221918,  0.25201793,
        0.2529571 ,  0.25389628,  0.25537212,  0.25691504,  0.25886047,
        0.25268877,  0.25416461,  0.25450003,  0.25429878,  0.25476837,
        0.25570754,  0.25624421,  0.2571163 ,  0.25919589,  0.2608059 ,
        0.24832833,  0.25020667,  0.25121293,  0.25060917,  0.25054209,
        0.25087751,  0.25235335,  0.2542317 ,  0.25611004,  0.25785422,
        0.25201793,  0.25329252,  0.25275585,  0.2525546 ,  0.2529571 ,
        0.25362794,  0.25496962,  0.25631129,  0.2583238 ,  0.25986673,
        0.32118126,  0.31353371,  0.30340407,  0.29890946,  0.29602486,
        0.2970982 ,  0.29233525,  0.28804189,  0.25047501,  0.23437491,
        0.32433419,  0.31353371,  0.30340407,  0.29890946,  0.29991571,
        0.29763487,  0.29233525,  0.28804189,  0.25047501,  0.25329252,
        0.31460705,  0.30420907,  0.29924488,  0.29220109,  0.29602486,
        0.29649444,  0.29233525,  0.2525546 ,  0.23269782,  0.23

In [97]:
env.state.shape

(138,)